# Классификация автоматизированного трафика Авито

**Задача**: по событиям внутри суточного окна оценить вероятность того, что `cookie_id` принадлежит трафику известных сервисов сбора данных. Основная метрика - максимальный precision при recall не ниже 70% на уровне куки.

Импортируем необходимые библиотеки. Фиксируем константы и SEED.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

from metric import precision_at_recall

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

DATA_DIR = Path('data')
SEED = 42
np.random.seed(SEED)

## 1. Проверка данных и временного окна

Сначала проверяем схему, уникальность куки и состав примера ответа. Затем оставляем только события с `window_start_ts <= event_ts < window_end_ts`. Полные дубликаты удаляются после фильтрации времени, разные события в одну секунду сохраняются.

Загружаем данные

In [ ]:
date_cols = ['cookie_created_at', 'window_start_ts', 'window_end_ts']
train = pd.read_csv(DATA_DIR / 'train.csv', parse_dates=date_cols)
test = pd.read_csv(DATA_DIR / 'test.csv', parse_dates=date_cols)
events = pd.read_csv(DATA_DIR / 'events.csv', parse_dates=['event_ts'])
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')


Минимальная проверка данных

In [ ]:
# Проверяем, что каждой куке соответствует одно окно.
assert train.cookie_id.is_unique and test.cookie_id.is_unique
assert set(train.cookie_id).isdisjoint(test.cookie_id)
assert (train.window_start_ts < train.window_end_ts).all()
assert (test.window_start_ts < test.window_end_ts).all()

meta = pd.concat([train, test], ignore_index=True)
assert set(events.cookie_id).issubset(meta.cookie_id)
print(f'Train: {len(train)} кук; test: {len(test)} кук')


In [ ]:
# Оставляем события внутри индивидуального окна cookie.
windowed = events.merge(
    meta[['cookie_id', 'window_start_ts', 'window_end_ts']],
    on='cookie_id',
    validate='many_to_one',
)
in_window = (
    windowed.event_ts.ge(windowed.window_start_ts)
    & windowed.event_ts.lt(windowed.window_end_ts)
)
windowed = windowed.loc[in_window]

In [ ]:
# Удаляем только полные дубликаты событий.
clean_events = windowed.drop_duplicates(
    subset=events.columns.tolist()
).copy()

# Приводим названия платформ к единым значениям.
clean_events['platform'] = (
    clean_events.platform.str.casefold()
    .replace({'desktop': 'web', 'iphone': 'ios'})
)
clean_events = clean_events.sort_values(
    ['cookie_id', 'event_ts'], kind='stable'
).reset_index(drop=True)

In [ ]:
summary = pd.Series({
    'train cookies': len(train),
    'test cookies': len(test),
    'positive train cookies': int(train.target.sum()),
    'all events': len(events),
    'events inside window': len(windowed),
    'full duplicates inside window': len(windowed) - len(clean_events),
    'clean events': len(clean_events),
})
display(summary.to_frame('count'))


In [ ]:
# Проверка на вхождение в интервал окна наблюдения.
assert clean_events.event_ts.ge(clean_events.window_start_ts).all()
assert clean_events.event_ts.lt(clean_events.window_end_ts).all()
assert clean_events.event_name.ne('captcha_shown').all()


### Вывод по этапу №1:

Для проверки моделей используем временные разбиения по дате начала окна window_start_ts. 
- На A обучаемся на 6-9 апреля и проверяем на 10-12 апреля; 
- На B обучаемся на 6-12 апреля и проверяем на 13-16 апреля; 
- На C обучаемся на 6-16 апреля и проверяем на 17-19 апреля. 

Это позволяет оценивать качество на более поздних куки, не подмешивая их в обучение.

## 2. Baseline

Воспроизводим quickstart: два признака (`n_events`, `item_nunique`) и RandomForest с 300 деревьями. Обучение: 6–16 апреля; проверка: 17–19 апреля. Метрика берётся из `metric.py`.

In [ ]:
baseline_features = (
    windowed.groupby('cookie_id')
    .agg(
        n_events=('event_ts', 'size'),
        item_nunique=('item_id', 'nunique'),
    )
    .reset_index()
)

X_baseline = (
    train[['cookie_id']]
    .merge(baseline_features, on='cookie_id', how='left', validate='one_to_one')
    .fillna({'n_events': 0, 'item_nunique': 0})
)

In [ ]:
feature_cols = ['n_events', 'item_nunique']
split_date = pd.Timestamp('2026-04-17')
is_train = train.window_start_ts < split_date


In [ ]:
baseline = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=3,
    random_state=SEED,
)
baseline.fit(
    X_baseline.loc[is_train, feature_cols],
    train.loc[is_train, 'target'],
)

y_valid = train.loc[~is_train, 'target']
score_valid = baseline.predict_proba(
    X_baseline.loc[~is_train, feature_cols]
)[:, 1]

In [ ]:
baseline_results = pd.DataFrame([{
    'precision_at_recall_70': precision_at_recall(y_valid, score_valid),
    'average_precision': average_precision_score(y_valid, score_valid),
    'roc_auc': roc_auc_score(y_valid, score_valid),
}])
display(baseline_results)


При seed=42 результат на проверке 17–19 апреля: **P@R≥0.70 = 0.106161**, Average Precision = 0.180385, ROC-AUC = 0.608129.

## 3. Поведенческие признаки

Собираем одну строку на куку только из `clean_events` внутри её окна. Признаки группировал так: состав действий, время, сессии, устройство и поиск. Целевая переменная в расчёт не передаётся.

In [ ]:
# Общая таблица: одна строка на cookie_id, без target.
features = meta[['cookie_id']].set_index('cookie_id')
cookie_id_groupby = clean_events.groupby('cookie_id', sort=False)
features['n_events'] = cookie_id_groupby.size()


In [ ]:
# Сколько раз встретился каждый тип действия и какова его доля.
event_counts = pd.crosstab(clean_events.cookie_id, clean_events.event_name)


In [ ]:
for event_name in sorted(clean_events.event_name.unique()):
    features[f'n_{event_name}'] = event_counts[event_name]
    features[f'share_{event_name}'] = features[f'n_{event_name}'] / features.n_events


Возраст куки, число заполненных полей и разные интересы могут отличать массовый сбор объявлений от обычного просмотра. Доли считаем относительно числа наблюдаемых значений, чтобы пропуск не выглядел как отдельное действие.

In [ ]:
meta_by_cookie = meta.set_index('cookie_id')

features['cookie_age_start_days'] = (
    meta_by_cookie.window_start_ts - meta_by_cookie.cookie_created_at
).dt.total_seconds() / 86400
features['cookie_age_end_days'] = (
    meta_by_cookie.window_end_ts - meta_by_cookie.cookie_created_at
).dt.total_seconds() / 86400
features['created_in_window'] = (
    meta_by_cookie.cookie_created_at.ge(meta_by_cookie.window_start_ts)
).astype(int)

In [ ]:
# Отдельно храним объём и разнообразие действий.
columns = [
    'item_id', 'item_category', 'item_location', 'search_query',
    'seller_type', 'user_agent', 'platform',
]
for column in columns:
    features[f'n_{column}_present'] = cookie_id_groupby[column].count()
    features[f'n_{column}_unique'] = cookie_id_groupby[column].nunique()
    features[f'unique_{column}_share'] = (
        features[f'n_{column}_unique']
        / features[f'n_{column}_present'].replace(0, np.nan)
    )

In [ ]:
item_views = clean_events.loc[clean_events.event_name.eq('item_view')]
features['n_viewed_items'] = item_views.groupby('cookie_id').item_id.nunique()
features['repeat_item_view_share'] = (
    (features.n_item_view - features.n_viewed_items)
    / features.n_item_view.replace(0, np.nan)
)

Время измеряем относительно границ окна. Выделим количество активных часов и плотность событий в одной минуте.

In [ ]:
first_event = cookie_id_groupby.event_ts.min()
last_event = cookie_id_groupby.event_ts.max()

features['activity_span_s'] = (last_event - first_event).dt.total_seconds()
features['first_event_delay_s'] = (
    first_event - meta_by_cookie.window_start_ts
).dt.total_seconds()
features['last_event_before_end_s'] = (
    meta_by_cookie.window_end_ts - last_event
).dt.total_seconds()

# Сколько разных часов и минут кука была активна.
events_by_cookie = clean_events.groupby('cookie_id').event_ts
features['active_hours'] = events_by_cookie.agg(
    lambda s: s.dt.floor('h').nunique()
)
features['active_minutes'] = events_by_cookie.agg(
    lambda s: s.dt.floor('min').nunique()
)

In [ ]:
minute_counts = clean_events.groupby(
    ['cookie_id', clean_events.event_ts.dt.floor('min')]
).size()
features['max_events_per_minute'] = minute_counts.groupby(level=0).max()
features['top_minute_share'] = (
    features.max_events_per_minute / features.n_events
)

Теперь посчитаем паузы между действиями. Интервалы считаем после сортировки событий внутри куки. Нулевые интервалы выделяем отдельно: при одинаковой секунде точный порядок действий неизвестен. Для куки с одним событием статистика интервалов остаётся пропущенной.

In [ ]:
# Паузы считаем между соседними событиями одной куки.
clean_events['gap_s'] = cookie_id_groupby.event_ts.diff().dt.total_seconds()
gap_stats = cookie_id_groupby.gap_s.agg(
    ['median', 'mean', 'std', 'min', 'max']
)
for stat in gap_stats.columns:
    features[f'gap_{stat}_s'] = gap_stats[stat]

features['gap_q10_s'] = cookie_id_groupby.gap_s.quantile(0.10)
features['gap_q90_s'] = cookie_id_groupby.gap_s.quantile(0.90)
features['gap_cv'] = (
    features.gap_std_s / features.gap_mean_s.replace(0, np.nan)
)
features['n_gaps'] = cookie_id_groupby.gap_s.count()

In [ ]:
# Частые короткие паузы и точные совпадения времени считаем отдельно.
for seconds in [0, 1, 5, 10]:
    short = clean_events.gap_s.eq(0) if seconds == 0 else clean_events.gap_s.le(seconds)
    features[f'gap_le_{seconds}_share'] = (
        short.groupby(clean_events.cookie_id).sum()
        / features.n_gaps.replace(0, np.nan)
    )


Новой сессией считаем действие после паузы более 30 минут. Благодаря этому длинный перерыв между визитами не смешивается с ритмом действий внутри одного визита.

In [ ]:
# Пауза более 30 минут начинает новую сессию.
new_session = clean_events.gap_s.isna() | clean_events.gap_s.gt(1800)
clean_events['session_no'] = (
    new_session.groupby(clean_events.cookie_id).cumsum()
)
sessions = clean_events.groupby(['cookie_id', 'session_no']).agg(
    n_events=('event_ts', 'size'),
    first=('event_ts', 'min'),
    last=('event_ts', 'max'),
)
sessions['span_s'] = (sessions['last'] - sessions['first']).dt.total_seconds()

features['n_sessions'] = sessions.groupby(level=0).size()
features['max_events_per_session'] = sessions.n_events.groupby(level=0).max()
features['median_events_per_session'] = (
    sessions.n_events.groupby(level=0).median()
)
features['max_session_span_s'] = sessions.span_s.groupby(level=0).max()

Теперь считаем паузы между событиями внутри одной сессии и статистики по всей куки из всех ее сессий.

In [ ]:
within_gap = clean_events.gap_s.where(
    clean_events.gap_s.gt(0) & clean_events.gap_s.le(1800)
)
within_stats = within_gap.groupby(clean_events.cookie_id).agg(
    ['count', 'median', 'mean', 'std']
)

features['n_within_gaps'] = within_stats['count']
features['within_gap_median_s'] = within_stats['median']
features['within_gap_mean_s'] = within_stats['mean']
features['within_gap_cv'] = (
    within_stats['std'] / within_stats['mean'].replace(0, np.nan)
)

Платформу и User-Agent сводим к простым группам. Сырая строка User-Agent и точные версии браузеров не используем. Наличие курсора оцениваем только среди web-событий.

In [ ]:
platform_counts = pd.crosstab(clean_events.cookie_id, clean_events.platform)


Тут считаем доли платформ и семейства User-Agent. Категории привязываем к cookie_id. Выделим следующие: 'http_client', 'headless', 'avito_app', 'yandex', 'firefox', 'chrome'.

In [ ]:
features['platform_main'] = platform_counts.idxmax(axis=1)
for platform in ['web', 'android', 'ios']:
    features[f'platform_{platform}_share'] = (
        platform_counts.get(platform, 0) / features.n_events
    )

ua = cookie_id_groupby.user_agent.first().str.lower()
ua_family = np.select(
    [
        ua.str.contains('python|scrapy|curl|go-http-client|node-fetch'),
        ua.str.contains('headless'),
        ua.str.startswith('avito/'),
        ua.str.contains('yabrowser'),
        ua.str.contains('firefox'),
        ua.str.contains('chrome'),
    ],
    ['http_client', 'headless', 'avito_app', 'yandex', 'firefox', 'chrome'],
    default='other',
)

# np.select возвращает массив без индекса cookie_id.
features['ua_family'] = pd.Series(ua_family, index=ua.index)

То же самое сделаем и с операционными системами.

In [ ]:
os_family = np.select(
    [
        ua.str.contains('android'),
        ua.str.contains('iphone|ios'),
        ua.str.contains('windows'),
        ua.str.contains('macintosh'),
        ua.str.contains('linux'),
    ],
    ['android', 'ios', 'windows', 'mac', 'linux'],
    default='other',
)

features['os_family'] = pd.Series(os_family, index=ua.index)

# Проверяем, что категории остались у соответствующих cookies.
assert features.loc[ua.str.startswith('avito/'), 'ua_family'].eq('avito_app').all()
assert features.loc[ua.str.contains('android'), 'os_family'].eq('android').all()

Проверим, была ли смена User-Agent или нет, а также количество смен

In [ ]:
features['ua_changes'] = (
    clean_events.user_agent.ne(cookie_id_groupby.user_agent.shift())
    .groupby(clean_events.cookie_id).sum() - 1
)


Также отдельно разберем группу web. Посчитаем количество событий, число событий с заполненным pointer_x (то есть с координатой) и их долю среди web-событий.

In [ ]:
web_events = clean_events.loc[clean_events.platform.eq('web')]
features['n_web_events'] = web_events.groupby('cookie_id').size()
features['n_web_pointer'] = web_events.groupby('cookie_id').pointer_x.count()
features['web_pointer_share'] = (
    features.n_web_pointer
    / features.n_web_events.replace(0, np.nan)
)

In [ ]:
# Это редкие координаты в событиях, а не полная траектория мыши.
features['pointer_x_std'] = web_events.groupby('cookie_id').pointer_x.std()
features['pointer_y_std'] = web_events.groupby('cookie_id').pointer_y.std()

features['n_unique_pointer_positions'] = (
    web_events[['cookie_id', 'pointer_x', 'pointer_y']]
    .dropna().drop_duplicates().groupby('cookie_id').size()
)


Дальше проанализируем поиск. Для поиска важны глубина просмотра выдачи и число поисковых действий. Пропуски `search_page` у других типов событий здесь не считаются признаком бота.

In [ ]:
search = clean_events.loc[
    clean_events.event_name.eq('search_results_view')
]
search_group = search.groupby('cookie_id').search_page
features['search_page_max'] = search_group.max()
features['search_page_median'] = search_group.median()
features['search_page_q90'] = search_group.quantile(0.90)

for page in [3, 5, 10]:
    features[f'search_page_gt_{page}_share'] = (
        search.search_page.gt(page).groupby(search.cookie_id).sum()
        / features.n_search_results_view.replace(0, np.nan)
    )

features['seller_pro_share'] = (
    clean_events.seller_type.eq('pro').groupby(clean_events.cookie_id).sum()
    / features.n_seller_type_present.replace(0, np.nan)
)

Обработаем пропуски, бесконечные значений (могли возникнуть при делении на ноль), выделяем категориальные признаки

In [ ]:
features = features.replace([np.inf, -np.inf], np.nan)

count_columns = [
    'n_viewed_items', 'n_web_events', 'n_web_pointer',
    'n_unique_pointer_positions',
]
features[count_columns] = features[count_columns].fillna(0)

cat_cols = ['platform_main', 'ua_family', 'os_family']
features[cat_cols] = features[cat_cols].fillna('MISSING').astype(str)

In [ ]:
assert len(features) == len(meta)
assert features.index.is_unique
assert np.isfinite(features.select_dtypes(include='number').fillna(0).to_numpy()).all()

print(f'Признаков: {features.shape[1]}; куки: {features.shape[0]}')


### Дополнительные признаки

Добавляем характеристики пауз, сессий, концентрации действий и движения указателя.

In [ ]:
features['gap_q25_s'] = cookie_id_groupby.gap_s.quantile(0.25)
features['gap_q75_s'] = cookie_id_groupby.gap_s.quantile(0.75)
features['gap_iqr_s'] = features.gap_q75_s - features.gap_q25_s

gap_counts = clean_events.dropna(subset=['gap_s']).groupby(
    ['cookie_id', 'gap_s']
).size()
features['most_common_gap_share'] = (
    gap_counts.groupby(level=0).max()
    / features.n_gaps.replace(0, np.nan)
)

In [ ]:
gap_bins = pd.cut(
    clean_events.gap_s,
    [-1, 0, 1, 5, 10, 30, 60, 300, 1800, np.inf],
)
gap_bin_counts = clean_events.groupby(
    ['cookie_id', gap_bins], observed=True
).size()
gap_bin_prob = (
    gap_bin_counts / gap_bin_counts.groupby(level=0).transform('sum')
)
features['gap_bin_entropy'] = (
    -(gap_bin_prob * np.log(gap_bin_prob)).groupby(level=0).sum()
)

In [ ]:
features['median_session_span_s'] = sessions.span_s.groupby(level=0).median()
features['mean_events_per_session'] = sessions.n_events.groupby(level=0).mean()
features['singleton_session_share'] = (
    sessions.n_events.eq(1).groupby(level=0).sum()
    / features.n_sessions
)
features['night_event_share'] = (
    clean_events.event_ts.dt.hour.lt(6).groupby(clean_events.cookie_id).sum()
    / features.n_events
)

Концентрацию по категориям, локациям и запросам считаем только по непустым значениям.

In [ ]:
for column in ['item_category', 'item_location', 'search_query']:
    counts = clean_events.dropna(subset=[column]).groupby(
        ['cookie_id', column]
    ).size()
    probability = counts / counts.groupby(level=0).transform('sum')

    features[f'{column}_top_share'] = (
        counts.groupby(level=0).max()
        / features[f'n_{column}_present'].replace(0, np.nan)
    )
    features[f'{column}_entropy'] = (
        -(probability * np.log(probability)).groupby(level=0).sum()
    )

Смены User-Agent считаем между разными секундами. Секунды с несколькими значениями исключаем, так как порядок событий внутри секунды неизвестен.

In [ ]:
ua_seconds = (
    clean_events.groupby(['cookie_id', 'event_ts'])
    .user_agent.agg(['first', 'nunique'])
    .reset_index()
)
features['ua_conflicted_seconds'] = (
    ua_seconds['nunique'].gt(1).groupby(ua_seconds.cookie_id).sum()
)

ua_seconds = ua_seconds.loc[ua_seconds['nunique'].eq(1)].copy()
features['ua_changes'] = (
    ua_seconds['first'].ne(ua_seconds.groupby('cookie_id')['first'].shift())
    .groupby(ua_seconds.cookie_id).sum() - 1
)

Между доступными точками указателя в разные секунды измеряем расстояние. Это не непрерывная траектория мыши.

In [ ]:
pointer = web_events[
    ['cookie_id', 'event_ts', 'pointer_x', 'pointer_y']
].dropna().copy()
pointer_groups = pointer.groupby(['cookie_id', 'event_ts'])

ambiguous = pointer_groups[['pointer_x', 'pointer_y']].nunique().max(axis=1).gt(1)
pointer = pointer_groups.first().loc[~ambiguous].reset_index()
previous_point = pointer.groupby('cookie_id')[
    ['pointer_x', 'pointer_y']
].shift()

pointer['move_px'] = np.hypot(
    pointer.pointer_x - previous_point.pointer_x,
    pointer.pointer_y - previous_point.pointer_y,
)
pointer_moves = pointer.groupby('cookie_id').move_px
features['n_pointer_moves'] = pointer_moves.count()
features['pointer_move_median_px'] = pointer_moves.median()
features['pointer_move_q90_px'] = pointer_moves.quantile(0.90)

In [ ]:
# Нулём обозначаем отсутствие наблюдений только у счётчиков.
features = features.replace([np.inf, -np.inf], np.nan)
constant_cols = features.columns[features.nunique(dropna=False).eq(1)].tolist()
features = features.drop(columns=constant_cols).copy()

for column in ['n_pointer_moves', 'ua_changes', 'ua_conflicted_seconds']:
    features[column] = features[column].fillna(0)

assert features.index.equals(pd.Index(meta.cookie_id))
assert np.isfinite(features.select_dtypes(include='number').fillna(0).to_numpy()).all()

print(f'Признаков: {features.shape[1]}; удалены константы: {constant_cols}')


### Вывод по этапу №3

Для каждой куки собрали одну строку из 110 признаков: состав и частоту действий, разнообразие объявлений и запросов, возраст куки, время активности, паузы и сессии, платформу и User-Agent, глубину поиска и характеристики доступных координат указателя. Признаки рассчитаны только по событиям внутри окна наблюдения, без использования target.

## 4. Первая модель CatBoost

Сравниваем CatBoost с quickstart baseline на одних датах: обучение до 13 апреля, проверка 13–16 апреля. Основная метрика берётся из metric.py.

In [ ]:
X_train = features.loc[train.cookie_id].reset_index(drop=True)
X_test = features.loc[test.cookie_id].reset_index(drop=True)

y = train.target.reset_index(drop=True)

valid_from = pd.Timestamp('2026-04-13')
valid_to = pd.Timestamp('2026-04-17')

fit_rows = train.window_start_ts.lt(valid_from).to_numpy()
valid_rows = train.window_start_ts.between(valid_from, valid_to, inclusive='left').to_numpy()

y_B = y.loc[valid_rows]

print(f'Train: {fit_rows.sum()}, valid: {valid_rows.sum()}, bots: {y_B.sum()}')


In [ ]:
# Baseline на тех же датах B.
baseline_B = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=SEED)

baseline_B.fit(X_baseline.loc[fit_rows, feature_cols], y.loc[fit_rows])

baseline_score_B = baseline_B.predict_proba(X_baseline.loc[valid_rows, feature_cols])[:, 1]


In [ ]:
model_B = CatBoostClassifier(
    iterations=1200, depth=5, learning_rate=0.05, l2_leaf_reg=10,
    loss_function='Logloss', random_seed=SEED, thread_count=4,
    allow_writing_files=False, verbose=False,
)

model_B.fit(
    X_train.loc[fit_rows], y.loc[fit_rows], cat_features=cat_cols,
    eval_set=(X_train.loc[valid_rows], y_B), early_stopping_rounds=150,
)

catboost_score_B = model_B.predict_proba(X_train.loc[valid_rows])[:, 1]

print(f'Лучшее число деревьев: {model_B.best_iteration_ + 1}')


In [ ]:
comparison_B = pd.DataFrame({
    'model': ['quickstart_baseline_B', 'catboost_ua_fixed_B'],
    'precision_at_recall_70': [
        precision_at_recall(y_B, baseline_score_B),
        precision_at_recall(y_B, catboost_score_B),
    ],
    'average_precision': [
        average_precision_score(y_B, baseline_score_B),
        average_precision_score(y_B, catboost_score_B),
    ],
    'roc_auc': [
        roc_auc_score(y_B, baseline_score_B),
        roc_auc_score(y_B, catboost_score_B),
    ],
})

display(comparison_B)


Давайте посмотрим на важность признаков обученной модели.

In [ ]:
importance_B = pd.Series(
    model_B.feature_importances_, index=X_train.columns
).sort_values(ascending=False)
display(importance_B.head(20).to_frame('importance'))

Среди первых важных - разброс pointer_x, разнообразие объявлений, движения указателя и паузы. Но сами по себе эти признаки не доказывают, что каждый из них улучшает результат.

### Вывод по этапу №4

После исправления UA/OS на разбиении B CatBoost получил **P@R≥0.70 = 0.7073** против 0.0967 у quickstart baseline. Average Precision = 0.7610, ROC-AUC = 0.9312. Ранняя остановка выбрала 729 деревьев; для отдельной проверки исправления ниже оставляем 675 деревьев исходной модели.

## 5. CatBoost после исправления признаков

Сохраняем исходные параметры и 675 деревьев, чтобы проверить влияние только исправленных UA/OS. Сначала оценим модель на A, B и C, затем обучим её на всём train и сохраним прогноз отдельно от уже отправленного v1.

### Проверка исправленного кандидата

Для каждого разбиения обучаемся только на более ранних датах. Параметры модели во всех трёх случаях одинаковые. Период C уже использовался в предыдущих экспериментах и служит дополнительной проверкой.

In [ ]:
corrected_params = dict(
    iterations=675,
    depth=5,
    learning_rate=0.05,
    l2_leaf_reg=10,
    loss_function='Logloss',
    random_seed=SEED,
    thread_count=4,
    allow_writing_files=False,
    verbose=False,
)

In [ ]:
validation_periods = [
    ('A', '2026-04-10', '2026-04-13'),
    ('B', '2026-04-13', '2026-04-17'),
    ('C', '2026-04-17', '2026-04-20'),
]
corrected_rows = []

for name, period_start, period_end in validation_periods:
    fit_mask = train.window_start_ts.lt(period_start)
    valid_mask = train.window_start_ts.between(
        period_start, period_end, inclusive='left'
    )

    model = CatBoostClassifier(**corrected_params)
    model.fit(X_train.loc[fit_mask], y.loc[fit_mask], cat_features=cat_cols)
    scores = model.predict_proba(X_train.loc[valid_mask])[:, 1]

    corrected_rows.append({
        'split': name,
        'precision_at_recall_70': precision_at_recall(y.loc[valid_mask], scores),
        'average_precision': average_precision_score(y.loc[valid_mask], scores),
    })

corrected_validation = pd.DataFrame(corrected_rows)
display(corrected_validation)

In [ ]:
final_ua_fixed = CatBoostClassifier(**corrected_params)
final_ua_fixed.fit(X_train, y, cat_features=cat_cols)

In [ ]:
# Привязываем прогноз к ID, чтобы сохранить порядок шаблона ответа.
test_scores = pd.Series(
    final_ua_fixed.predict_proba(X_test)[:, 1],
    index=test.cookie_id,
)
submission_ua_fixed = sample_submission[['cookie_id']].copy()
submission_ua_fixed['score'] = submission_ua_fixed.cookie_id.map(test_scores)

assert submission_ua_fixed.cookie_id.equals(sample_submission.cookie_id)
assert np.isfinite(submission_ua_fixed.score).all()


In [ ]:
Path('artifacts').mkdir(exist_ok=True)
submission_ua_fixed.to_csv('artifacts/submission_ua_fixed.csv', index=False)
print(f'Сохранено {len(submission_ua_fixed)} строк')


## 6. Отбор признаков

В предыдущем эксперименте сравнили 110 исходных признаков, сокращение до 97 и расширение до 138. Эти результаты были получены до исправления UA/OS. Ниже повторяем сравнение 110 и 97 признаков на исправленной таблице; генерацию полного набора v2 в итоговый ноутбук не возвращаем.

| Набор | Признаков | B: P@R≥0.70 | A: P@R≥0.70 |
|---|---:|---:|---:|
| Исправленный полный | 110 | 0.7073 | 0.7273 |
| Без UA/platform | 97 | 0.7008 | 0.7306 |

Для сравнения наборов ниже используем одинаковые параметры и 729 деревьев, выбранных на B. Проверенный ранее полный набор v2 (138 признаков) в этих новых условиях не оценивался.

In [ ]:
# Убираем только техническую группу, которую проверили на A и B.
technical_prefixes = (
    'ua_', 'n_user_agent', 'unique_user_agent', 'os_family',
    'platform_', 'n_platform', 'unique_platform',
)

technical_cols = [
    column for column in features.columns
    if column.startswith(technical_prefixes)
]

selected_cols = [
    column for column in features.columns
    if column not in technical_cols
]


In [ ]:
assert len(features.columns) == 110 and len(selected_cols) == 97
print(f'Исключено {len(technical_cols)} признаков; осталось {len(selected_cols)}')


Сравниваем v1 и сокращённый набор на двух временных разбиениях. Число деревьев и остальные параметры одинаковые, чтобы оценить именно состав признаков.

In [ ]:
stage3_iterations = model_B.best_iteration_ + 1

fit_A = train.window_start_ts.lt('2026-04-10').to_numpy()
valid_A = train.window_start_ts.between('2026-04-10', '2026-04-13', inclusive='left').to_numpy()

stage3_splits = {
    'B': (fit_rows, valid_rows),
    'A': (fit_A, valid_A),
}

feature_sets = {
    'v1': features.columns.tolist(),
    'v1_no_ua': selected_cols,
}


In [ ]:
def predict_on_split(columns, train_rows, validation_rows):
    model = CatBoostClassifier(
        iterations=stage3_iterations,
        depth=5,
        learning_rate=0.05,
        l2_leaf_reg=10,
        loss_function='Logloss',
        random_seed=SEED,
        thread_count=4,
        allow_writing_files=False,
        verbose=False,
    )
    model.fit(
        X_train.loc[train_rows, columns],
        y.loc[train_rows],
        cat_features=[column for column in cat_cols if column in columns],
    )
    return model.predict_proba(
        X_train.loc[validation_rows, columns]
    )[:, 1]

In [ ]:
stage3_rows = []

for split_name, (train_rows, validation_rows) in stage3_splits.items():
    target = y.loc[validation_rows]

    for name, columns in feature_sets.items():
        scores = predict_on_split(columns, train_rows, validation_rows)
        stage3_rows.append({
            'experiment': f'{name}_{split_name}',
            'precision_at_recall_70': precision_at_recall(target, scores),
            'average_precision': average_precision_score(target, scores),
            'roc_auc': roc_auc_score(target, scores),
            'n_features': len(columns),
        })

stage3_results = pd.DataFrame(stage3_rows)

display(stage3_results)


### Вывод по этапу отбора признаков

После исправления UA/OS полный набор немного лучше на B (0.7073 против 0.7008), сокращённый до 97 признаков — немного лучше на A (0.7306 против 0.7273). Однозначного преимущества сокращения нет. Дальнейшие эксперименты с весами класса на 97 признаках ниже относятся к отдельной модели.

## 7. Настройка CatBoost

Проверяем пять заранее выбранных конфигураций на двух наборах признаков. На B подбираем параметры и число деревьев по редкой сетке; лучшие варианты перепроверяем на A. Контроль C пока не используем.

In [ ]:
tuning_sets = {
    'v1': features.columns.tolist(),
    'no_ua': selected_cols,
}

tuning_configs = [
    {'depth': 4, 'learning_rate': 0.05, 'l2_leaf_reg': 10},
    {'depth': 5, 'learning_rate': 0.05, 'l2_leaf_reg': 10},
    {'depth': 6, 'learning_rate': 0.05, 'l2_leaf_reg': 10},
    {'depth': 7, 'learning_rate': 0.03, 'l2_leaf_reg': 30},
    {'depth': 5, 'learning_rate': 0.08, 'l2_leaf_reg': 3},
]

print(f'Обучений на B: {len(tuning_sets) * len(tuning_configs)}')


In [ ]:
def fit_on_B(columns, params):
    model = CatBoostClassifier(
        iterations=1600,
        loss_function='Logloss',
        random_seed=SEED,
        thread_count=4,
        allow_writing_files=False,
        verbose=False,
        **params,
    )

    model.fit(
        X_train.loc[fit_rows, columns],
        y.loc[fit_rows],
        cat_features=[column for column in cat_cols if column in columns],
        eval_set=(X_train.loc[valid_rows, columns], y_B),
        early_stopping_rounds=200,
        use_best_model=False,
    )
    return model

Для каждой обученной модели измеряем официальную метрику и Average Precision примерно через каждые 100 деревьев. Поскольку это грубая сетка: точный максимум после каждого дерева был бы слишком шумным на 248 ботах B.

In [ ]:
checkpoint_rows = []
checkpoint_scores_B = {}

for set_name, columns in tuning_sets.items():
    for config_id, params in enumerate(tuning_configs):
        model = fit_on_B(columns, params)
        checkpoints = list(range(100, model.tree_count_ + 1, 100))
        checkpoints += [model.best_iteration_ + 1, model.tree_count_]

        for trees in sorted(set(checkpoints)):
            scores = model.predict_proba(
                X_train.loc[valid_rows, columns],
                ntree_end=trees,
            )[:, 1]
            key = (set_name, config_id, trees)
            checkpoint_scores_B[key] = scores

            checkpoint_rows.append({
                'feature_set': set_name,
                'config_id': config_id,
                'trees': trees,
                'precision_at_recall_70': precision_at_recall(y_B, scores),
                'average_precision': average_precision_score(y_B, scores),
            })

        print(set_name, config_id, 'деревьев:', model.tree_count_)

In [ ]:
checkpoint_B = pd.DataFrame(checkpoint_rows)
best_by_config_B = (
    checkpoint_B
    .sort_values(
        ['precision_at_recall_70', 'average_precision'],
        ascending=False,
    )
    .drop_duplicates(['feature_set', 'config_id'])
)
display(best_by_config_B)

На A проверяем по два лучших варианта каждого набора. Число деревьев уже выбрано на B и здесь не меняется.

In [ ]:
finalists_B = (
    best_by_config_B
    .sort_values(
        ['feature_set', 'precision_at_recall_70'],
        ascending=[True, False],
    )
    .groupby('feature_set').head(2)
    .reset_index(drop=True)
)
display(finalists_B)

In [ ]:
finalist_rows_A = []
finalist_scores_A = {}
finalist_scores_B = {}

for row in finalists_B.itertuples():
    columns = tuning_sets[row.feature_set]
    params = tuning_configs[row.config_id]
    model = CatBoostClassifier(
        iterations=int(row.trees),
        loss_function='Logloss',
        random_seed=SEED,
        thread_count=4,
        allow_writing_files=False,
        verbose=False,
        **params,
    )
    model.fit(
        X_train.loc[fit_A, columns],
        y.loc[fit_A],
        cat_features=[column for column in cat_cols if column in columns],
    )

    key = (row.feature_set, row.config_id)
    scores_A = model.predict_proba(X_train.loc[valid_A, columns])[:, 1]
    finalist_scores_A[key] = scores_A
    finalist_scores_B[key] = checkpoint_scores_B[
        (row.feature_set, row.config_id, row.trees)
    ]

    finalist_rows_A.append({
        'feature_set': row.feature_set,
        'config_id': row.config_id,
        'trees': row.trees,
        'precision_A': precision_at_recall(y.loc[valid_A], scores_A),
        'average_precision_A': average_precision_score(y.loc[valid_A], scores_A),
        'precision_B': row.precision_at_recall_70,
    })

In [ ]:
finalists_AB = pd.DataFrame(finalist_rows_A)
display(finalists_AB.sort_values('precision_B', ascending=False))

### Веса класса

Проверяем веса ботов 2 и 4 для перспективного CatBoost без UA/platform. Сравниваем результаты на тех же временных проверках B и A.

In [ ]:
weight_scores = {
    1: {
        'A': finalist_scores_A[('no_ua', 3)],
        'B': finalist_scores_B[('no_ua', 3)],
    }
}
weight_params = tuning_configs[3]
weight_trees = int(
    finalists_B.loc[
        finalists_B.feature_set.eq('no_ua')
        & finalists_B.config_id.eq(3),
        'trees',
    ].iloc[0]
)
print('Опорная модель:', weight_params, 'деревьев:', weight_trees)

In [ ]:
for bot_weight in [2, 4]:
    weight_scores[bot_weight] = {}

    for split_name, (train_rows, validation_rows) in stage3_splits.items():
        model = CatBoostClassifier(
            iterations=weight_trees,
            loss_function='Logloss',
            class_weights=[1, bot_weight],
            random_seed=SEED,
            thread_count=4,
            allow_writing_files=False,
            verbose=False,
            **weight_params,
        )
        model.fit(X_train.loc[train_rows, selected_cols], y.loc[train_rows])
        weight_scores[bot_weight][split_name] = model.predict_proba(
            X_train.loc[validation_rows, selected_cols]
        )[:, 1]

    print('Вес бота:', bot_weight)

In [ ]:
weight_rows = []
for bot_weight, split_scores in weight_scores.items():
    for split_name, scores in split_scores.items():
        target = y.loc[stage3_splits[split_name][1]]
        weight_rows.append({
            'bot_weight': bot_weight,
            'split': split_name,
            'precision_at_recall_70': precision_at_recall(target, scores),
            'average_precision': average_precision_score(target, scores),
        })

weight_results = pd.DataFrame(weight_rows)
display(weight_results)

### Выбор для контрольного периода C

Фиксируем два рецепта CatBoost на 97 признаках: depth=7, learning_rate=0.03, l2_leaf_reg=30, 1000 деревьев, seed=42; веса бота 4 и 2. Вес 4 дал P@R≥0.70 0.713 на B и 0.805 на A; вес 2 - 0.719 и 0.769. Отправленный v1 остаётся резервом с известным скрытым результатом 0.78589.

In [ ]:
stage4_recipes = {
    f'weighted_{bot_weight}': {
        'columns': selected_cols,
        'iterations': weight_trees,
        'depth': weight_params['depth'],
        'learning_rate': weight_params['learning_rate'],
        'l2_leaf_reg': weight_params['l2_leaf_reg'],
        'class_weights': [1, bot_weight],
    }
    for bot_weight in [4, 2]
}

## 8. Финальная проверка на C

Кандидаты и число деревьев зафиксированы по A/B. Обучаем их на 6-16 апреля и проверяем на 17-19 апреля. C уже использовался для оценки quickstart baseline, поэтому это контроль кандидатов, а не полностью независимый holdout.

In [ ]:
control_from = pd.Timestamp('2026-04-17')
control_train = train.window_start_ts.lt(control_from).to_numpy()
control_valid = ~control_train
y_C = y.loc[control_valid]

print(
    f'Обучение: {control_train.sum()} кук; '
    f'C: {control_valid.sum()} кук, из них {y_C.sum()} ботов'
)

Для основной метрики берём лучший допустимый порог с учётом равных score. Число TP/FP и дневные срезы относятся к этому порогу и служат диагностикой; на скрытом тесте порог будет другим.

In [ ]:
def best_operating_point(target, score):
    # Сортируем score и объединяем одинаковые значения в одну точку.
    values = np.asarray(score)
    labels = np.asarray(target)
    order = np.argsort(-values, kind='mergesort')
    values, labels = values[order], labels[order]
    ends = np.r_[values[1:] != values[:-1], True]

    tp = np.cumsum(labels)[ends]
    predicted = np.arange(1, len(labels) + 1)[ends]
    precision = tp / predicted
    recall = tp / labels.sum()

    eligible = np.flatnonzero(recall >= 0.70)
    best = eligible[np.argmax(precision[eligible])]
    return {
        'precision': precision[best],
        'recall': recall[best],
        'tp': int(tp[best]),
        'fp': int(predicted[best] - tp[best]),
        'threshold': values[ends][best],
    }

In [ ]:
control_scores = {}
for name, recipe in stage4_recipes.items():
    params = {key: value for key, value in recipe.items() if key != 'columns'}
    model = CatBoostClassifier(
        **params,
        loss_function='Logloss',
        random_seed=SEED,
        thread_count=4,
        allow_writing_files=False,
        verbose=False,
    )
    model.fit(
        X_train.loc[control_train, recipe['columns']],
        y.loc[control_train],
    )
    control_scores[name] = model.predict_proba(
        X_train.loc[control_valid, recipe['columns']]
    )[:, 1]
    print(name, 'готов')

In [ ]:
control_rows = []
for name, scores in control_scores.items():
    point = best_operating_point(y_C, scores)
    control_rows.append({
        'model': name,
        'precision_at_recall_70': point['precision'],
        'recall': point['recall'],
        'tp': point['tp'],
        'fp': point['fp'],
        'average_precision': average_precision_score(y_C, scores),
        'roc_auc': roc_auc_score(y_C, scores),
        'threshold': point['threshold'],
    })

control_results = pd.DataFrame(control_rows)
display(control_results)

In [ ]:
# Дневные срезы считаем при одном пороге для каждой модели.
control_predictions = train.loc[
    control_valid, ['cookie_id', 'window_start_ts', 'target']
].copy()
for name, scores in control_scores.items():
    control_predictions[name] = scores

In [ ]:
daily_rows = []
for row in control_results.itertuples():
    for day, part in control_predictions.groupby(
        control_predictions.window_start_ts.dt.date
    ):
        predicted = part[row.model].ge(row.threshold)
        tp = int((predicted & part.target.eq(1)).sum())
        fp = int((predicted & part.target.eq(0)).sum())
        daily_rows.append({
            'model': row.model,
            'day': day,
            'bots': int(part.target.sum()),
            'tp': tp,
            'fp': fp,
            'precision': tp / (tp + fp) if tp + fp else np.nan,
            'recall': tp / part.target.sum(),
        })

daily_results = pd.DataFrame(daily_rows)
display(daily_results)

Если результаты близки, оцениваем разброс их разницы парным bootstrap: в каждой повторной выборке обе модели получают одни и те же куки. Он показывает неопределённость на 160 ботах C.

In [ ]:
positive = np.flatnonzero(y_C.to_numpy() == 1)
negative = np.flatnonzero(y_C.to_numpy() == 0)
rng = np.random.default_rng(SEED)
bootstrap_deltas = []

for _ in range(500):
    rows = np.r_[
        rng.choice(positive, len(positive), replace=True),
        rng.choice(negative, len(negative), replace=True),
    ]
    target = y_C.to_numpy()[rows]
    p4 = precision_at_recall(target, control_scores['weighted_4'][rows])
    p2 = precision_at_recall(target, control_scores['weighted_2'][rows])
    bootstrap_deltas.append(p4 - p2)

print('95% интервал разницы вес 4 − вес 2:',
      np.quantile(bootstrap_deltas, [0.025, 0.975]))

Дополнительная проверка: обучаем выбранный рецепт только на последней неделе до C. Это будет тестом зависимости от длины обучающей истории.

In [ ]:
# Правило выбора было задано до просмотра C.
margin = abs(
    control_results.iloc[0].precision_at_recall_70
    - control_results.iloc[1].precision_at_recall_70
)
chosen_name = (
    'weighted_4' if margin < 0.02
    else control_results.sort_values(
        'precision_at_recall_70', ascending=False
    ).iloc[0].model
)
chosen_recipe = stage4_recipes[chosen_name]
print('Выбранный рецепт:', chosen_name)

In [ ]:
week_train = train.window_start_ts.between(
    '2026-04-10', control_from, inclusive='left'
).to_numpy()
week_params = {
    key: value for key, value in chosen_recipe.items()
    if key != 'columns'
}
week_model = CatBoostClassifier(
    **week_params,
    loss_function='Logloss',
    random_seed=SEED,
    thread_count=4,
    allow_writing_files=False,
    verbose=False,
)
week_model.fit(
    X_train.loc[week_train, chosen_recipe['columns']],
    y.loc[week_train],
)
week_score = week_model.predict_proba(
    X_train.loc[control_valid, chosen_recipe['columns']]
)[:, 1]
print('Неделя:', precision_at_recall(y_C, week_score))
print('Вся история:', precision_at_recall(y_C, control_scores[chosen_name]))

In [ ]:
# Сравниваем группы при пороге, найденном на всём C.
control_predictions['n_events'] = X_train.loc[
    control_valid, 'n_events'
].to_numpy()
control_predictions['platform'] = X_train.loc[
    control_valid, 'platform_main'
].to_numpy()
threshold = control_results.set_index('model').loc[chosen_name, 'threshold']

segments = {
    '1–6 событий': control_predictions.n_events.le(6),
    'более 6 событий': control_predictions.n_events.gt(6),
    'mobile': control_predictions.platform.isin(['android', 'ios']),
    'web': control_predictions.platform.eq('web'),
}

In [ ]:
segment_rows = []
for name, mask in segments.items():
    part = control_predictions.loc[mask]
    predicted = part[chosen_name].ge(threshold)
    tp = int((predicted & part.target.eq(1)).sum())
    fp = int((predicted & part.target.eq(0)).sum())

    segment_rows.append({
        'segment': name,
        'cookies': len(part),
        'bots': int(part.target.sum()),
        'tp': tp,
        'fp': fp,
        'recall': tp / part.target.sum(),
        'precision': tp / (tp + fp) if tp + fp else np.nan,
    })

segment_results = pd.DataFrame(segment_rows)
display(segment_results)

### Вывод по контрольному периоду

На C вариант с весом бота 2 получил P@R≥0.70 = 0.757 (112 TP, 36 FP), вес 4 - 0.703 (116 TP, 49 FP). Парный bootstrap дал для разницы «вес 4 минус вес 2» интервал от -0.090 до +0.027. Обучение только на последней неделе снизило результат веса 2 до 0.696. На коротких траекториях и мобильном трафике recall ниже.

По C для второй онлайн-попытки выбрали вес 2. На скрытом тесте он получил 0.77088 и уступил исходному v1 (0.78589). Это показывает, что локальная временная проверка полезна, но не гарантирует порядок моделей на более позднем test.

## 9. Результаты отправок на Stepik

| Отправка | Модель | Скрытая P@R≥0.70 |
|---|---|---:|
| v1 | CatBoost, 110 признаков до исправления UA/OS | 0.78589 |
| v2 | CatBoost, 97 признаков, вес бота 2 | 0.77088 |
| v3 | v1 с 1200 деревьями | 0.78019 |
| v4 | CatBoost после исправления UA/OS и Optuna | **0.81566** |
| v5 | XGBoost после исправления UA/OS и Optuna | 0.80637 |

Лучшим подтверждённым вариантом остаётся v4. Ноутбук сохраняет его в `submission.csv`; прежние прогнозы находятся в `artifacts/`. После пяти отправок остаются две попытки.

In [ ]:
submission = submission_ua_fixed[['cookie_id', 'score']].copy()
assert submission.cookie_id.equals(sample_submission.cookie_id)
assert np.isfinite(submission.score).all()

submission.to_csv('submission.csv', index=False)
print(f'Готово: submission.csv, {len(submission)} строк')

## 10. Сравнение LightGBM и XGBoost

Сравниваем два умеренно регуляризованных варианта каждой модели с CatBoost на исправленных 110 признаках. Временные разбиения и метрика совпадают с предыдущей проверкой. C уже использовался ранее, поэтому служит дополнительной проверкой устойчивости.

In [ ]:
# Одинаково кодируем категории в train и test.
X_boost = X_train.copy()
X_boost_test = X_test.copy()

for column in cat_cols:
    X_boost[column] = X_boost[column].astype('category')
    X_boost_test[column] = pd.Categorical(
        X_boost_test[column], categories=X_boost[column].cat.categories
    )


Для LightGBM проверяем два размера дерева. Ограничиваем число листьев и минимальный размер листа, чтобы маленькая выборка не приводила к слишком дробным правилам.

In [ ]:
lgbm_configs = {
    'LightGBM_15': dict(
        n_estimators=700, learning_rate=0.05, num_leaves=15,
        max_depth=5, min_child_samples=40, reg_lambda=10,
    ),
    'LightGBM_7': dict(
        n_estimators=900, learning_rate=0.035, num_leaves=7,
        max_depth=4, min_child_samples=60, reg_lambda=15,
    ),
}

for params in lgbm_configs.values():
    params.update(random_state=SEED, n_jobs=4, verbosity=-1)

Для XGBoost также проверяем более глубокий и более компактный вариант. Категориальные признаки обрабатываются самой библиотекой.

In [ ]:
xgb_configs = {
    'XGBoost_depth4': dict(
        n_estimators=700, learning_rate=0.05, max_depth=4,
        min_child_weight=5, reg_lambda=10,
    ),
    'XGBoost_depth3': dict(
        n_estimators=1000, learning_rate=0.035, max_depth=3,
        min_child_weight=7, reg_lambda=15,
    ),
}

for params in xgb_configs.values():
    params.update(
        tree_method='hist', enable_categorical=True,
        subsample=0.9, colsample_bytree=0.9,
        eval_metric='logloss', random_state=SEED, n_jobs=4,
    )

Обучаем каждый вариант только на датах до начала проверочного периода. Сохраняем официальный precision и Average Precision для каждого разбиения.

In [ ]:
boosting_candidates = [
    (name, LGBMClassifier, params)
    for name, params in lgbm_configs.items()
] + [
    (name, XGBClassifier, params)
    for name, params in xgb_configs.items()
]

catboost_rows = [
    {
        'model': 'CatBoost_675',
        'split': row.split,
        'precision_at_recall_70': row.precision_at_recall_70,
        'average_precision': row.average_precision,
    }
    for row in corrected_validation.itertuples()
]

In [ ]:
boosting_rows = catboost_rows.copy()

for model_name, model_type, params in boosting_candidates:
    for split_name, period_start, period_end in validation_periods:
        fit_mask = train.window_start_ts.lt(period_start)
        valid_mask = train.window_start_ts.between(
            period_start, period_end, inclusive='left'
        )

        model = model_type(**params)
        model.fit(X_boost.loc[fit_mask], y.loc[fit_mask])
        scores = model.predict_proba(X_boost.loc[valid_mask])[:, 1]

        boosting_rows.append({
            'model': model_name,
            'split': split_name,
            'precision_at_recall_70': precision_at_recall(y.loc[valid_mask], scores),
            'average_precision': average_precision_score(y.loc[valid_mask], scores),
        })

boosting_results = pd.DataFrame(boosting_rows)

In [ ]:
precision_table = boosting_results.pivot(
    index='model', columns='split', values='precision_at_recall_70'
)
ap_table = boosting_results.pivot(
    index='model', columns='split', values='average_precision'
)

display(precision_table.round(4))
display(ap_table.round(4))

### Вывод по сравнению моделей

XGBoost с глубиной 4 улучшил P@R≥0.70 на A (0.7619 против 0.7407) и B (0.7190 против 0.6944), а также Average Precision на всех трёх разбиениях. На C его P@R≥0.70 составила 0.7107 против 0.7278 у CatBoost. Ни один вариант не улучшил официальную метрику одновременно на A, B и C. Для возможной онлайн-проверки сохраняем XGBoost отдельно от текущего `submission.csv`.

In [ ]:
xgb_final = XGBClassifier(**xgb_configs['XGBoost_depth4'])
xgb_final.fit(X_boost, y)

In [ ]:
xgb_scores = pd.Series(
    xgb_final.predict_proba(X_boost_test)[:, 1],
    index=test.cookie_id,
)
submission_xgb = sample_submission[['cookie_id']].copy()
submission_xgb['score'] = submission_xgb.cookie_id.map(xgb_scores)

assert submission_xgb.cookie_id.equals(sample_submission.cookie_id)
assert np.isfinite(submission_xgb.score).all()

In [ ]:
submission_xgb.to_csv('artifacts/submission_xgb_depth4.csv', index=False)
print(f'Сохранено {len(submission_xgb)} строк в artifacts/submission_xgb_depth4.csv')

## 11. Подбор XGBoost с Optuna

Подбираем параметры XGBoost на A и B. Целевая функция — среднее значение официальной метрики с небольшим штрафом за разницу между разбиениями. Average Precision отслеживаем отдельно. Поиск ограничен 40 испытаниями и одним часом; SQLite в `artifacts/` позволяет продолжить прерванный запуск.

In [ ]:
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

xgb_common = dict(
    tree_method='hist', enable_categorical=True,
    eval_metric='logloss', random_state=SEED, n_jobs=4,
)

search_periods = validation_periods[:2]
search_splits = [
    (
        name,
        train.window_start_ts.lt(start),
        train.window_start_ts.between(start, end, inclusive='left'),
    )
    for name, start, end in search_periods
]

Ищем рядом с лучшей проверенной конфигурацией. Первым испытанием ставим именно её, чтобы поиск не потерял исходную точку сравнения.

In [ ]:
def suggest_xgb_params(trial):
    return dict(
        n_estimators=trial.suggest_int('n_estimators', 450, 1100, step=50),
        learning_rate=trial.suggest_float('learning_rate', 0.025, 0.09, log=True),
        max_depth=trial.suggest_int('max_depth', 3, 6),
        min_child_weight=trial.suggest_float('min_child_weight', 2, 12, log=True),
        reg_lambda=trial.suggest_float('reg_lambda', 2, 30, log=True),
        gamma=trial.suggest_float('gamma', 0, 2),
        subsample=trial.suggest_float('subsample', 0.75, 1.0),
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.75, 1.0),
    )

In [ ]:
def xgb_objective(trial):
    params = {**xgb_common, **suggest_xgb_params(trial)}
    precisions = []

    for name, fit_mask, valid_mask in search_splits:
        model = XGBClassifier(**params)
        model.fit(X_boost.loc[fit_mask], y.loc[fit_mask])
        scores = model.predict_proba(X_boost.loc[valid_mask])[:, 1]

        precision = precision_at_recall(y.loc[valid_mask], scores)
        precisions.append(precision)
        trial.set_user_attr(f'precision_{name}', precision)
        trial.set_user_attr(
            f'ap_{name}', average_precision_score(y.loc[valid_mask], scores)
        )

    return np.mean(precisions) - 0.25 * (max(precisions) - min(precisions))

In [ ]:
study = optuna.create_study(
    study_name='xgb_ua_fixed_v1',
    storage='sqlite:///artifacts/optuna_xgb.sqlite3',
    load_if_exists=True,
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED),
)

if not study.trials:
    study.enqueue_trial({
        **{key: xgb_configs['XGBoost_depth4'][key] for key in [
            'n_estimators', 'learning_rate', 'max_depth',
            'min_child_weight', 'reg_lambda', 'subsample',
            'colsample_bytree',
        ]},
        'gamma': 0.0,
    })

completed = sum(
    trial.state == optuna.trial.TrialState.COMPLETE
    for trial in study.trials
)
remaining = max(0, 40 - completed)
print(f'Готово испытаний: {completed}; осталось: {remaining}')

Каждое испытание обучает модель дважды — для A и B. Промежуточное отсечение не используем: целевую метрику можно посчитать только после получения всех прогнозов на проверочных куках.

In [ ]:
def show_search_progress(study, trial):
    if (trial.number + 1) % 10 == 0:
        print(f'Готово {trial.number + 1} испытаний; лучшая цель: {study.best_value:.4f}')


if remaining:
    study.optimize(
        xgb_objective,
        n_trials=remaining,
        timeout=3600,
        callbacks=[show_search_progress],
    )

print(f'Испытаний завершено: {len(study.trials)}')
print(f'Лучшее значение цели: {study.best_value:.4f}')

In [ ]:
ranked_trials = sorted(
    [trial for trial in study.trials if trial.value is not None],
    key=lambda trial: trial.value,
    reverse=True,
)

trial_rows = [
    {
        'trial': trial.number,
        'objective': trial.value,
        'precision_A': trial.user_attrs['precision_A'],
        'precision_B': trial.user_attrs['precision_B'],
        'ap_A': trial.user_attrs['ap_A'],
        'ap_B': trial.user_attrs['ap_B'],
    }
    for trial in ranked_trials[:5]
]
display(pd.DataFrame(trial_rows).round(4))
print('Лучшие параметры:', study.best_params)

Три лучших варианта по A/B дополнительно проверяем на более позднем C. C не участвовал в подборе параметров, но уже использовался в прежних экспериментах.

In [ ]:
c_fit = train.window_start_ts.lt('2026-04-17')
c_valid = train.window_start_ts.between(
    '2026-04-17', '2026-04-20', inclusive='left'
)

c_rows = []
for trial in ranked_trials[:3]:
    model = XGBClassifier(**xgb_common, **trial.params)
    model.fit(X_boost.loc[c_fit], y.loc[c_fit])
    scores = model.predict_proba(X_boost.loc[c_valid])[:, 1]

    c_rows.append({
        'trial': trial.number,
        'precision_A': trial.user_attrs['precision_A'],
        'precision_B': trial.user_attrs['precision_B'],
        'precision_C': precision_at_recall(y.loc[c_valid], scores),
        'ap_C': average_precision_score(y.loc[c_valid], scores),
    })

c_results = pd.DataFrame(c_rows)
display(c_results.round(4))

### Вывод по подбору

Среди 40 испытаний лучшая конфигурация по A/B получила P@R≥0.70 **0.7547 / 0.7404**, но на C — **0.7000**. Более ровная среди трёх лидеров получила **0.7767 / 0.7220 / 0.7273** на A/B/C. Сохраняем оба прогноза отдельно: первый выбран по критерию поиска, второй — по наименьшему результату на трёх периодах. Период C уже участвовал в предыдущих экспериментах, поэтому его оценка не независима.

In [ ]:
# Среди трёх лидеров выбираем вариант с лучшим худшим периодом.
stable_row = c_results.assign(
    worst=c_results[['precision_A', 'precision_B', 'precision_C']].min(axis=1)
).sort_values('worst', ascending=False).iloc[0]

stable_trial = next(
    trial for trial in ranked_trials if trial.number == int(stable_row.trial)
)
candidate_trials = {
    'ab': study.best_trial,
    'stable': stable_trial,
}

for name, trial in candidate_trials.items():
    print(f'{name}: испытание {trial.number}')

Обучаем оба варианта на всех размеченных куках и возвращаем прогноз в порядке `sample_submission.csv`. Текущий `submission.csv` остаётся прежним до решения об отправке.

In [ ]:
for name, trial in candidate_trials.items():
    model = XGBClassifier(**xgb_common, **trial.params)
    model.fit(X_boost, y)

    scores = pd.Series(
        model.predict_proba(X_boost_test)[:, 1],
        index=test.cookie_id,
    )
    submission = sample_submission[['cookie_id']].copy()
    submission['score'] = submission.cookie_id.map(scores)

    assert submission.score.notna().all()
    assert submission.score.between(0, 1).all()

    output_path = f'artifacts/submission_xgb_optuna_{name}.csv'
    submission.to_csv(output_path, index=False)
    print(f'Сохранено {len(submission)} строк: {output_path}')

## 12. Подбор CatBoost с Optuna

Подбираем параметры на исправленных 110 признаках. Исходный CatBoost с 675 деревьями включён первым испытанием. Цель — высокая P@R≥0.70 одновременно на A и B; дополнительно следим за Average Precision. Ограничиваем поиск 32 испытаниями и 75 минутами. Результаты сохраняются в SQLite, поэтому поиск можно продолжить после остановки.

In [ ]:
cat_common = dict(
    loss_function='Logloss',
    random_seed=SEED,
    thread_count=4,
    allow_writing_files=False,
    verbose=False,
)

cat_search_splits = [
    (
        name,
        train.window_start_ts.lt(start),
        train.window_start_ts.between(start, end, inclusive='left'),
    )
    for name, start, end in validation_periods[:2]
]

Проверяем умеренную область параметров вокруг исходной модели. Для байесовского bootstrap отдельно подбираем его интенсивность; вариант `Default` оставляет стандартный способ сэмплирования CatBoost.

In [ ]:
def suggest_cat_params(trial):
    trial.suggest_int('iterations', 400, 1100, step=25)
    trial.suggest_int('depth', 4, 7)
    trial.suggest_float('learning_rate', 0.025, 0.09, log=True)
    trial.suggest_float('l2_leaf_reg', 3, 30, log=True)
    trial.suggest_float('random_strength', 0, 2)
    trial.suggest_float('rsm', 0.75, 1.0)

    bootstrap = trial.suggest_categorical(
        'bootstrap_type', ['Default', 'Bayesian']
    )
    if bootstrap == 'Bayesian':
        trial.suggest_float('bagging_temperature', 0, 2)


def cat_params(trial):
    params = trial.params.copy()
    bootstrap = params.pop('bootstrap_type')

    if bootstrap == 'Bayesian':
        params['bootstrap_type'] = bootstrap

    return {**cat_common, **params}

In [ ]:
def cat_objective(trial):
    suggest_cat_params(trial)
    params = cat_params(trial)
    precisions = []

    for name, fit_mask, valid_mask in cat_search_splits:
        model = CatBoostClassifier(**params)
        model.fit(
            X_train.loc[fit_mask], y.loc[fit_mask], cat_features=cat_cols
        )
        scores = model.predict_proba(X_train.loc[valid_mask])[:, 1]

        precision = precision_at_recall(y.loc[valid_mask], scores)
        precisions.append(precision)
        trial.set_user_attr(f'precision_{name}', precision)
        trial.set_user_attr(
            f'ap_{name}', average_precision_score(y.loc[valid_mask], scores)
        )

    return np.mean(precisions) - 0.25 * (max(precisions) - min(precisions))

In [ ]:
cat_study = optuna.create_study(
    study_name='catboost_ua_fixed_v1',
    storage='sqlite:///artifacts/optuna_catboost.sqlite3',
    load_if_exists=True,
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED),
)

if not cat_study.trials:
    cat_study.enqueue_trial({
        'iterations': 675,
        'depth': 5,
        'learning_rate': 0.05,
        'l2_leaf_reg': 10,
        'random_strength': 1.0,
        'rsm': 1.0,
        'bootstrap_type': 'Default',
    })

cat_completed = sum(
    trial.state == optuna.trial.TrialState.COMPLETE
    for trial in cat_study.trials
)
cat_remaining = max(0, 32 - cat_completed)
print(f'Готово испытаний: {cat_completed}; осталось: {cat_remaining}')

In [ ]:
def show_cat_progress(study, trial):
    if (trial.number + 1) % 8 == 0:
        print(f'Готово {trial.number + 1}; лучшая цель: {study.best_value:.4f}')


if cat_remaining:
    cat_study.optimize(
        cat_objective,
        n_trials=cat_remaining,
        timeout=4500,
        callbacks=[show_cat_progress],
    )

print(f'Завершено испытаний: {len(cat_study.trials)}')
print(f'Лучшая цель: {cat_study.best_value:.4f}')

In [ ]:
cat_ranked = sorted(
    [trial for trial in cat_study.trials if trial.value is not None],
    key=lambda trial: trial.value,
    reverse=True,
)

cat_rows = [
    {
        'trial': trial.number,
        'objective': trial.value,
        'precision_A': trial.user_attrs['precision_A'],
        'precision_B': trial.user_attrs['precision_B'],
        'ap_A': trial.user_attrs['ap_A'],
        'ap_B': trial.user_attrs['ap_B'],
    }
    for trial in cat_ranked[:5]
]
display(pd.DataFrame(cat_rows).round(4))
print('Лучшие параметры:', cat_study.best_params)

Три лучшие конфигурации по A/B проверяем на C, который не участвовал в подборе. Однако C уже использовался в прежних экспериментах, поэтому это лишь диагностическая оценка.

In [ ]:
cat_c_fit = train.window_start_ts.lt('2026-04-17')
cat_c_valid = train.window_start_ts.between(
    '2026-04-17', '2026-04-20', inclusive='left'
)

cat_c_rows = []
for trial in cat_ranked[:3]:
    model = CatBoostClassifier(**cat_params(trial))
    model.fit(
        X_train.loc[cat_c_fit], y.loc[cat_c_fit], cat_features=cat_cols
    )
    scores = model.predict_proba(X_train.loc[cat_c_valid])[:, 1]

    cat_c_rows.append({
        'trial': trial.number,
        'precision_A': trial.user_attrs['precision_A'],
        'precision_B': trial.user_attrs['precision_B'],
        'precision_C': precision_at_recall(y.loc[cat_c_valid], scores),
        'ap_C': average_precision_score(y.loc[cat_c_valid], scores),
    })

cat_c_results = pd.DataFrame(cat_c_rows)
display(cat_c_results.round(4))

### Вывод по подбору CatBoost

Лучший по A/B вариант получил P@R≥0.70 **0.7265 / 0.7190 / 0.7019** на A/B/C. Среди трёх лидеров более устойчивым оказался другой вариант: **0.7547 / 0.7068 / 0.7386**. Он превысил исходный CatBoost на каждом периоде (**0.7407 / 0.6944 / 0.7278**). Сохраняем оба прогноза отдельно; C уже использовался в прежних экспериментах и не является независимым тестом.

In [ ]:
# Выбираем самый сильный худший период среди трёх лидеров.
cat_stable_row = cat_c_results.assign(
    worst=cat_c_results[['precision_A', 'precision_B', 'precision_C']].min(axis=1)
).sort_values('worst', ascending=False).iloc[0]

cat_stable_trial = next(
    trial for trial in cat_ranked if trial.number == int(cat_stable_row.trial)
)
cat_candidates = {
    'ab': cat_study.best_trial,
    'stable': cat_stable_trial,
}

for name, trial in cat_candidates.items():
    print(f'{name}: испытание {trial.number}')

Переобучаем оба варианта на всех размеченных куках и возвращаем прогнозы в порядке `sample_submission.csv`. Вариант `stable` подтвердил на Stepik 0.81566, поэтому также сохраняем его в корневой `submission.csv`.

In [ ]:
for name, trial in cat_candidates.items():
    model = CatBoostClassifier(**cat_params(trial))
    model.fit(X_train, y, cat_features=cat_cols)

    scores = pd.Series(
        model.predict_proba(X_test)[:, 1],
        index=test.cookie_id,
    )
    submission = sample_submission[['cookie_id']].copy()
    submission['score'] = submission.cookie_id.map(scores)

    assert submission.score.notna().all()
    assert submission.score.between(0, 1).all()

    output_path = f'artifacts/submission_catboost_optuna_{name}.csv'
    submission.to_csv(output_path, index=False)
    if name == 'stable':
        submission.to_csv('submission.csv', index=False)
    print(f'Сохранено {len(submission)} строк: {output_path}')

## 13. Проверка новых блоков признаков

Проверили четыре дополнительных категориальных признака из событий внутри суточного окна: точную строку User-Agent и наиболее частые категорию, локацию и поисковый запрос. Модель и параметры CatBoost не меняли; сравнение на A/B/C:

| Блок | A | B | C |
|---|---:|---:|---:|
| Текущие 110 признаков | **0.7547** | **0.7068** | 0.7386 |
| + точный User-Agent | 0.7477 | 0.6984 | **0.7568** |
| + категория, локация, запрос | 0.7442 | 0.7045 | 0.7386 |
| + оба блока | 0.7512 | 0.6902 | 0.7517 |
| + указатель по типу события | 0.7767 | 0.6770 | 0.7179 |

Точный User-Agent улучшил только C. Статистика указателя по типу события улучшила A, но ухудшила B и C. Ни один новый блок не улучшил A и B одновременно, поэтому их не включаем в итоговый `submission.csv`. Подтверждённый результат Stepik для текущей модели остаётся **0.81566**.

## 14. Проверка компактных наборов признаков

Сначала обучаем лучший CatBoost только на датах до A и ранжируем 110 признаков по его важности. Так отбор не использует target проверочных периодов. Затем сравниваем 20, 40, 60 и все 110 признаков на одинаковых A/B/C без нового подбора параметров.

In [ ]:
rank_fit = train.window_start_ts.lt('2026-04-10')
rank_model = CatBoostClassifier(**cat_params(cat_stable_trial))
rank_model.fit(
    X_train.loc[rank_fit], y.loc[rank_fit], cat_features=cat_cols
)

feature_rank = pd.Series(
    rank_model.get_feature_importance(),
    index=X_train.columns,
).sort_values(ascending=False)

print(f'Для отбора использовано {rank_fit.sum()} кук')
display(feature_rank.head(20).round(2))

In [ ]:
feature_sets = {
    size: feature_rank.head(size).index.tolist()
    for size in [20, 40, 60]
}

for size, columns in feature_sets.items():
    selected_categories = [column for column in cat_cols if column in columns]
    print(f'{size} признаков; категории: {selected_categories}')

Каждый компактный набор переобучаем только на датах до своего проверочного периода. Полный набор уже проверен в предыдущем этапе, поэтому его результаты используем как контроль.

In [ ]:
compact_rows = []
for size, columns in feature_sets.items():
    selected_categories = [column for column in cat_cols if column in columns]

    for split_name, period_start, period_end in validation_periods:
        fit_mask = train.window_start_ts.lt(period_start)
        valid_mask = train.window_start_ts.between(
            period_start, period_end, inclusive='left'
        )

        model = CatBoostClassifier(**cat_params(cat_stable_trial))
        model.fit(
            X_train.loc[fit_mask, columns], y.loc[fit_mask],
            cat_features=selected_categories,
        )
        scores = model.predict_proba(
            X_train.loc[valid_mask, columns]
        )[:, 1]

        compact_rows.append({
            'features': size,
            'split': split_name,
            'precision': precision_at_recall(y.loc[valid_mask], scores),
            'ap': average_precision_score(y.loc[valid_mask], scores),
        })
        print(f'{size} признаков, {split_name}: '
              f'{compact_rows[-1]["precision"]:.4f}')

In [ ]:
base_row = cat_c_results.loc[
    cat_c_results.trial.eq(cat_stable_trial.number)
].iloc[0]
base_rows = [
    {'features': 110, 'split': split_name,
     'precision': base_row[f'precision_{split_name}'],
     'ap': ap}
    for split_name, ap in zip(
        ['A', 'B', 'C'],
        [cat_stable_trial.user_attrs['ap_A'],
         cat_stable_trial.user_attrs['ap_B'], base_row.ap_C],
    )
]

compact_results = pd.DataFrame([*compact_rows, *base_rows])
display(compact_results.pivot(index='features', columns='split', values='precision').round(4))
display(compact_results.pivot(index='features', columns='split', values='ap').round(4))

### Вывод по сокращению признаков

| Признаков | A | B | C |
|---:|---:|---:|---:|
| 20 | 0.7080 | 0.6468 | 0.6957 |
| 40 | 0.7175 | 0.6580 | **0.8058** |
| 60 | 0.7306 | 0.6824 | 0.7417 |
| 110 | **0.7547** | **0.7068** | 0.7386 |

Полный набор лучше на A и B. Результат 40 признаков на C интересен, но Average Precision почти не изменился: 0.7738 против 0.7727. Поэтому это экспериментальный кандидат, а не замена проверенному `submission.csv`.

In [ ]:
top40_columns = feature_sets[40]
top40_categories = [column for column in cat_cols if column in top40_columns]

top40_final = CatBoostClassifier(**cat_params(cat_stable_trial))
top40_final.fit(
    X_train[top40_columns], y, cat_features=top40_categories
)

scores = pd.Series(
    top40_final.predict_proba(X_test[top40_columns])[:, 1],
    index=test.cookie_id,
)
submission_top40 = sample_submission[['cookie_id']].copy()
submission_top40['score'] = submission_top40.cookie_id.map(scores)

assert submission_top40.score.notna().all()
assert submission_top40.score.between(0, 1).all()

submission_top40.to_csv('artifacts/submission_catboost_top40.csv', index=False)
print(f'Сохранено {len(submission_top40)} строк')

Дополнительная проверка 40 признаков внутри C: 17 апреля — 0.7581 против 0.7869 у полного набора; 18 апреля — 0.7755 против 0.7400; 19 апреля — 0.8108 против 0.7561. В стратифицированном bootstrap по кукам (500 повторов) 95% интервал разницы P@R≥0.70 составил **[−0.081; +0.111]**. Интервал включает ноль, а C уже использовался для выбора моделей, поэтому локальный выигрыш нельзя считать надёжным. `submission_catboost_top40.csv` сохраняется только как отдельный кандидат.

## 15. Нейросеть TabM

Проверяем [TabM](https://github.com/yandex-research/tabm) на тех же 110 признаках и временных окнах. Модель обучается локально на GPU, если он доступен. Числовые преобразования и словари категорий подгоняем только на обучающей части каждого разбиения.


In [ ]:
import torch
from rtdl_num_embeddings import PiecewiseLinearEmbeddings, compute_bins
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, QuantileTransformer
from tabm import TabM

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_cols = [column for column in X_train if column not in cat_cols]
print('Устройство:', device)


In [ ]:
def prepare_tabm(fit_frame, eval_frame):
    imputer = SimpleImputer(strategy='median', keep_empty_features=True)
    fit_num = imputer.fit_transform(fit_frame[num_cols])
    eval_num = imputer.transform(eval_frame[num_cols])

    quantiles = QuantileTransformer(
        n_quantiles=min(1000, len(fit_frame)),
        output_distribution='normal', random_state=SEED,
    )
    fit_num = quantiles.fit_transform(fit_num).astype('float32')
    eval_num = quantiles.transform(eval_num).astype('float32')

    encoder = OrdinalEncoder(
        handle_unknown='use_encoded_value', unknown_value=-1,
    )
    fit_cat = encoder.fit_transform(fit_frame[cat_cols]).astype('int64') + 1
    eval_cat = encoder.transform(eval_frame[cat_cols]).astype('int64') + 1

    return dict(
        fit_num=fit_num, eval_num=eval_num,
        fit_cat=fit_cat, eval_cat=eval_cat,
        cardinalities=[len(values) + 1 for values in encoder.categories_],
    )


In [ ]:
def predict_tabm(model, num, cat):
    model.eval()
    probabilities = []

    with torch.inference_mode():
        for start in range(0, len(num), 1024):
            x_num = torch.as_tensor(num[start:start + 1024], device=device)
            x_cat = torch.as_tensor(cat[start:start + 1024], device=device)
            logits = model(x_num, x_cat).squeeze(-1)
            probabilities.append(logits.sigmoid().mean(dim=1).cpu().numpy())

    return np.concatenate(probabilities)


In [ ]:
def train_tabm(data, target, max_epochs, valid_target=None, seed=SEED):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    bins = compute_bins(torch.from_numpy(data['fit_num']), n_bins=32)
    embeddings = PiecewiseLinearEmbeddings(
        bins, d_embedding=16, activation=False, version='B',
    )
    model = TabM.make(
        n_num_features=data['fit_num'].shape[1],
        cat_cardinalities=data['cardinalities'],
        num_embeddings=embeddings, d_out=1,
    ).to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=0.002, weight_decay=0.0003,
    )

    x_num = torch.as_tensor(data['fit_num'], device=device)
    x_cat = torch.as_tensor(data['fit_cat'], device=device)
    y_tensor = torch.tensor(target, dtype=torch.float32, device=device)
    best_ap, best_epoch, best_state, stale = -1.0, max_epochs, None, 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        for rows in torch.randperm(len(target), device=device).split(512):
            logits = model(x_num[rows], x_cat[rows]).squeeze(-1)
            labels = y_tensor[rows, None].expand_as(logits)
            loss = torch.nn.functional.binary_cross_entropy_with_logits(
                logits, labels,
            )
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

        if valid_target is not None:
            scores = predict_tabm(model, data['eval_num'], data['eval_cat'])
            ap = average_precision_score(valid_target, scores)
            if ap > best_ap + 1e-4:
                best_ap, best_epoch, stale = ap, epoch, 0
                best_state = {
                    key: value.cpu().clone()
                    for key, value in model.state_dict().items()
                }
            else:
                stale += 1
            if stale >= 15:
                break

        if epoch % 10 == 0 and valid_target is not None:
            print(f'Эпоха {epoch}; лучший AP: {best_ap:.4f}', flush=True)

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_epoch


Число эпох выбираем по Average Precision на B. Затем обучаем модели до A и C на столько же эпох; эти окна используются только для диагностики.


In [ ]:
fit_B = train.window_start_ts.lt('2026-04-13')
valid_B = train.window_start_ts.between(
    '2026-04-13', '2026-04-17', inclusive='left'
)

nn_B = prepare_tabm(X_train.loc[fit_B], X_train.loc[valid_B])
tabm_B, best_epoch = train_tabm(
    nn_B, y.loc[fit_B].to_numpy(), max_epochs=100,
    valid_target=y.loc[valid_B].to_numpy(),
)
score_B = predict_tabm(tabm_B, nn_B['eval_num'], nn_B['eval_cat'])
print('Лучших эпох:', best_epoch)
print('B P@R≥0.70:', precision_at_recall(y.loc[valid_B], score_B))
print('B AP:', average_precision_score(y.loc[valid_B], score_B))


In [ ]:
fit_A = train.window_start_ts.lt('2026-04-10')
valid_A = train.window_start_ts.between(
    '2026-04-10', '2026-04-13', inclusive='left'
)

nn_A = prepare_tabm(X_train.loc[fit_A], X_train.loc[valid_A])
tabm_A, _ = train_tabm(
    nn_A, y.loc[fit_A].to_numpy(), max_epochs=best_epoch,
)
score_A = predict_tabm(tabm_A, nn_A['eval_num'], nn_A['eval_cat'])
print('A P@R≥0.70:', precision_at_recall(y.loc[valid_A], score_A))
print('A AP:', average_precision_score(y.loc[valid_A], score_A))


In [ ]:
fit_C = train.window_start_ts.lt('2026-04-17')
valid_C = train.window_start_ts.between(
    '2026-04-17', '2026-04-20', inclusive='left'
)

nn_C = prepare_tabm(X_train.loc[fit_C], X_train.loc[valid_C])
tabm_C, _ = train_tabm(
    nn_C, y.loc[fit_C].to_numpy(), max_epochs=best_epoch,
)
score_C = predict_tabm(tabm_C, nn_C['eval_num'], nn_C['eval_cat'])
print('C P@R≥0.70:', precision_at_recall(y.loc[valid_C], score_C))
print('C AP:', average_precision_score(y.loc[valid_C], score_C))


### Результат нейросети

| Модель | A | B | C |
|---|---:|---:|---:|
| CatBoost, лучший на Stepik | **0.755** | 0.707 | 0.739 |
| TabM с числовыми эмбеддингами | 0.721 | **0.725** | **0.742** |

TabM лучше на B и C, но хуже на A. Усреднение трёх seed подняло B до 0.734, однако снизило C до 0.702. Смесь TabM с CatBoost тоже ухудшила C. Поэтому сохраняем отдельный прогноз одного TabM с seed 42; превосходство на скрытом тесте пока не подтверждено.


In [ ]:
# Число эпох выбрано на B; C не участвовал в выборе.
nn_full = prepare_tabm(X_train, X_test)
tabm_final, _ = train_tabm(
    nn_full, y.to_numpy(), max_epochs=best_epoch,
)
tabm_test_score = predict_tabm(
    tabm_final, nn_full['eval_num'], nn_full['eval_cat'],
)


In [ ]:
tabm_by_cookie = pd.Series(tabm_test_score, index=test.cookie_id)
submission_tabm = sample_submission[['cookie_id']].copy()
submission_tabm['score'] = submission_tabm.cookie_id.map(tabm_by_cookie)

assert submission_tabm.score.notna().all()
assert submission_tabm.score.between(0, 1).all()

output_path = 'artifacts/submission_tabm.csv'
submission_tabm.to_csv(output_path, index=False)
print(f'Сохранено {len(submission_tabm)} строк: {output_path}')
